# `runner.ipynb` — universal stage runner

**What this does**
1. Mounts Google Drive (Colab) so the cache + outputs persist.
2. Sources every code cell from `basic_cells.ipynb` (no copy-paste — single source of truth).
3. Reads one `configs/stage_X.yaml` and calls `run_stage(...)`.
4. Saves outputs under `<output_dir>/<stage_id>/` (config snapshot, predictions, metrics, plots, summary).

**To run a stage**
- Drop `configs/stage_X.yaml` into the project folder on Drive.
- Edit the `CONFIG_PATH` cell below.
- Runtime → Run all.

**To run a different stage in parallel**
- Open a second Colab session, same notebook file, point `CONFIG_PATH` at a different config. Cache is shared (Drive); outputs are namespaced by `stage_id`.

**First time only**
- Run the optional "bulk prefetch" cell once to fill the cache for *all* stages — softer on ISS than letting every stage prefetch its own slice on first run.

## 0. Locate the project on Drive

Set `PROJECT_DIR` to the folder containing this notebook + `basic_cells.ipynb` + `configs/`. Default assumes Colab with the project under `MyDrive/moex-hack/`.

In [2]:
import os
from pathlib import Path

IN_COLAB = "COLAB_GPU" in os.environ or "google.colab" in str(type(globals().get("get_ipython", lambda: None)()))
if IN_COLAB:
    from google.colab import drive
    if not os.path.ismount("/content/drive"):
        drive.mount("/content/drive", force_remount=False)
    PROJECT_DIR = "/content/drive/MyDrive/moex-hack"
else:
    PROJECT_DIR = str(Path.cwd())

os.chdir(PROJECT_DIR)
print(f"PROJECT_DIR = {PROJECT_DIR}")
print("contents:", sorted(os.listdir(PROJECT_DIR)))


Mounted at /content/drive
PROJECT_DIR = /content/drive/MyDrive/moex-hack
contents: ['.claude', '.git', '.gitignore', '.mcp.json', '.obsidian', '1B_res.md', '2510.15821v1.pdf', '345_res.md', 'ArenaGo test.md', 'Chronos2_Roma.ipynb', 'Clippings', 'Colab_user_tasks.md', 'LICENSE', 'Pasted image 20260525220744.png', 'Untitled 1.md', 'Untitled.md', 'answers.md', 'ass.json', 'basic_cells.ipynb', 'chat.md', 'configs', 'current_state.md', 'das', 'devdocs.md', 'engines.json', 'etolb', 'exp_plan.md', 'fetch_future_SiH6_24_2021-01-01_2025-12-31.parquet', 'ght.md', 'gp.md', 'index.md', 'm_arenago.txt', 'misha-gitlab-mhack.txt', 'moex_chronos2_pipeline.ipynb', 'my_questions.md', 'pr.md', 'pr_1.md', 'present.md', 'promt.md', 'repo_res', 'runner.ipynb', 'runs', 'scratchpads', 'sec.json', 'stage_B', 'team_repo', 'wiki.md', 'ые']


## 1. Source `basic_cells.ipynb`

Executes every code cell from the cell library inside this kernel — no imports, no module packaging, no version drift.

In [3]:
import json, nbformat
from IPython import get_ipython

BASIC = Path(PROJECT_DIR) / "basic_cells.ipynb"
assert BASIC.exists(), f"missing {BASIC} — copy it next to this notebook"
nb = nbformat.read(str(BASIC), as_version=4)
ip = get_ipython()
n_code = 0
for cell in nb.cells:
    if cell.cell_type == "code":
        ip.run_cell(cell.source)
        n_code += 1
print(f"sourced {n_code} code cells from basic_cells.ipynb")


/usr/local/lib/python3.12/dist-packages/nbformat/__init__.py:96: MissingIDFieldWarning: Cell is missing an id field, this will become a hard error in future nbformat versions. You may want to use `normalize()` on your notebooks before validations (available since nbformat 5.1.4). Previous versions of nbformat are fixing this issue transparently, and will stop doing so in the future.
  validate(nb)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.7/72.7 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 118.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 49.2 MB/s eta 0:00:00
GPU: Tesla T4  cap=(7, 5)  dtype=torch.float16
sourced 14 code cells from basic_cells.ipynb


## 2. (Optional, one-time) Bulk prefetch all stages

Run once on a fresh Drive cache. Idempotent — safe to re-run.

In [ ]:
# Uncomment to bulk-prefetch every configured stage's data into the cache.
all_cfgs = [load_config(str(p)) for p in sorted(Path("configs").glob("stage_*.yaml"))]
cache_dir = resolve_cache_dir(all_cfgs[0])
manifest = build_prefetch_manifest(all_cfgs)
print(f"manifest size: {len(manifest)} files")
prefetch_all(manifest, cache_dir)

cache_dir = /content/drive/MyDrive/moex_cache
manifest size: 822 files


prefetch:   0%|          | 0/822 [00:00<?, ?it/s]

  EMPTY futures/forts/BRF2 interval=24 [2024-01-01..2024-06-30]
  EMPTY futures/forts/BRG2 interval=24 [2024-01-01..2024-06-30]
  EMPTY futures/forts/BRH2 interval=24 [2024-01-01..2024-06-30]
  EMPTY futures/forts/BRJ2 interval=24 [2024-01-01..2024-06-30]
  EMPTY futures/forts/BRK2 interval=24 [2024-01-01..2024-06-30]
  EMPTY futures/forts/BRM2 interval=24 [2024-01-01..2024-06-30]
  EMPTY futures/forts/BRN2 interval=24 [2024-01-01..2024-06-30]
  EMPTY futures/forts/BRQ2 interval=24 [2024-01-01..2024-06-30]
  EMPTY futures/forts/BRU2 interval=24 [2024-01-01..2024-06-30]
  EMPTY futures/forts/BRV2 interval=24 [2024-01-01..2024-06-30]
  EMPTY futures/forts/BRX2 interval=24 [2024-01-01..2024-06-30]
  EMPTY futures/forts/BRZ2 interval=24 [2024-01-01..2024-06-30]
  EMPTY futures/forts/BRF3 interval=24 [2024-01-01..2024-06-30]
  EMPTY futures/forts/BRG3 interval=24 [2024-01-01..2024-06-30]
  EMPTY futures/forts/BRH3 interval=24 [2024-01-01..2024-06-30]
  EMPTY futures/forts/BRJ3 interval=24 [

## 3. Pick a stage and run

Edit `CONFIG_PATH` to point at the stage you want to run.

In [4]:
CONFIG_PATH = "configs/stage_1_daily.yaml"   # <-- edit me

summary = run_stage(CONFIG_PATH)


cache_dir = /content/drive/MyDrive/moex_cache


prefetch:   0%|          | 0/197 [00:00<?, ?it/s]

  CACHE     /content/drive/MyDrive/moex_cache/stock_shares_SBER_24_2021-01-01_2026-04-30.parquet  rows=1424
  CACHE     /content/drive/MyDrive/moex_cache/stock_shares_GAZP_24_2021-01-01_2026-04-30.parquet  rows=1424
  CACHE     /content/drive/MyDrive/moex_cache/stock_shares_LKOH_24_2021-01-01_2026-04-30.parquet  rows=1424
  CACHE     /content/drive/MyDrive/moex_cache/stock_shares_ROSN_24_2021-01-01_2026-04-30.parquet  rows=1424
  CACHE     /content/drive/MyDrive/moex_cache/stock_shares_NVTK_24_2021-01-01_2026-04-30.parquet  rows=1420
  CACHE     /content/drive/MyDrive/moex_cache/stock_shares_TATN_24_2021-01-01_2026-04-30.parquet  rows=1422
  CACHE     /content/drive/MyDrive/moex_cache/stock_shares_GMKN_24_2021-01-01_2026-04-30.parquet  rows=1420
  CACHE     /content/drive/MyDrive/moex_cache/stock_shares_PLZL_24_2021-01-01_2026-04-30.parquet  rows=1417
  CACHE     /content/drive/MyDrive/moex_cache/stock_shares_MAGN_24_2021-01-01_2026-04-30.parquet  rows=1424
  CACHE     /content/drive/M

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!
`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/478M [00:00<?, ?B/s]

walk-forward: 400 windows


windows:   0%|          | 0/400 [00:00<?, ?it/s]

SUMMARY: {
  "stage_id": "stage_1c_daily_ctx250",
  "n_windows": 400,
  "mean_da_primary": 0.4905555555555556,
  "median_da_primary": 0.48624999999999996,
  "cells_signif_05": 0,
  "mean_pearson_primary": -0.011704314780451986,
  "mean_coverage_primary": 0.798125
}


## 4. Inspect outputs in-line

After `run_stage` finishes, the cell below loads the metrics tables for quick review without leaving the notebook.

In [ ]:
cfg = load_config(CONFIG_PATH)
stage_out = Path(cfg["output_dir"]) / cfg["stage_id"]
print("outputs at:", stage_out)
print("files:", sorted(p.name for p in stage_out.rglob("*") if p.is_file()))

import pandas as pd
metrics = pd.read_csv(stage_out / "metrics.csv")
agg     = pd.read_csv(stage_out / "metrics_aggregate.csv")
print("\n--- aggregate ---");        print(agg.to_string(index=False))
print("\n--- per-cell (head) ---");  print(metrics.head(20).to_string(index=False))

# Display every plot saved by run_stage
from IPython.display import Image, display
for p in sorted((stage_out / "plots").glob("*.png")):
    print(p.name); display(Image(str(p)))


## 5. Notes

- **Cache only**: `run_stage` reads from Parquet cache. If a file is missing it errors loudly with the exact missing key — fix the config / re-run §2 prefetch.
- **Multiple stages in one session**: just re-edit `CONFIG_PATH` and re-run §3 + §4. Outputs are namespaced by `stage_id`; nothing is overwritten across stages.
- **Long runs**: `run_walk_forward` checkpoints `preds_partial.parquet` every 25 windows.
- **Path B**: when fine-tuning is wired in, it lives behind a `cfg["path_b"]["enabled"]` flag — same runner.